In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
for candidate in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
    if (candidate / "pyproject.toml").exists():
        PROJECT_ROOT = candidate
        break
sys.path.insert(0, str(PROJECT_ROOT / "src"))

CONFIG_PATH = "experiments/qwen3vl/qwen3vl_2b_qlora_r16_lr2e-4/config.toml"

from shield.config import load_and_validate, method

config = load_and_validate(PROJECT_ROOT / CONFIG_PATH)
print("Esperimento:", config["experiment"]["name"], "| method:", method(config))

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("Nessuna GPU disponibile. Verifica CUDA sul server.")
for index in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(index)
    print(f"GPU {index}: {props.name} - {props.total_memory / 1e9:.1f} GB")

torch.cuda.reset_peak_memory_stats()

In [ ]:
import time

from transformers import TrainerCallback


class LiveMonitorCallback(TrainerCallback):
    def __init__(self):
        self.t0 = None

    def on_train_begin(self, args, state, control, **kwargs):
        self.t0 = time.time()
        print(f"[monitor] training avviato · step totali previsti: {state.max_steps}")

    def _elapsed(self):
        return (time.time() - self.t0) / 60 if self.t0 else 0.0

    def _vram(self):
        return torch.cuda.max_memory_allocated() / 1e9 if torch.cuda.is_available() else 0.0

    def on_log(self, args, state, control, logs=None, **kwargs):
        logs = logs or {}
        if "loss" not in logs:  
            return
        lr = logs.get("learning_rate", float("nan"))
        print(
            f"[monitor] step {state.global_step:>5}/{state.max_steps}"
            f" · epoch {state.epoch or 0:5.2f}"
            f" · loss {logs['loss']:.4f}"
            f" · lr {lr:.2e}"
            f" · vram {self._vram():5.1f} GB"
            f" · {self._elapsed():5.1f} min",
            flush=True,
        )

    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        metrics = metrics or {}
        skip = {"eval_runtime", "eval_samples_per_second", "eval_steps_per_second"}
        evals = " · ".join(
            f"{k.replace('eval_', '')} {v:.4f}"
            for k, v in sorted(metrics.items())
            if k.startswith("eval_") and k not in skip and isinstance(v, (int, float))
        )
        print(
            f"[monitor] ── EVAL epoch {state.epoch or 0:5.2f} · {evals}"
            f" · vram {self._vram():5.1f} GB · {self._elapsed():5.1f} min",
            flush=True,
        )

In [ ]:
from shield.tracking import (
    log_artifact_if_exists,
    log_params,
    log_trainer_state,
    mlflow_run,
)
from shield.training.pipeline import run_training

result = None
if method(config) == "none":
    print("method='none' (baseline zero-shot): niente da addestrare.")
    print("Valuta il baseline con: scripts/evaluate/evaluate.py --config", CONFIG_PATH)
else:
    config_path = PROJECT_ROOT / CONFIG_PATH
    monitor = LiveMonitorCallback()
    with mlflow_run(config, root=PROJECT_ROOT):
        result = run_training(config, PROJECT_ROOT, callbacks=[monitor])
        log_params(result["trainable_summary"], prefix="model")
        log_trainer_state(str(result["trainer_state"]))
        log_artifact_if_exists(str(result["final_dir"]), artifact_path="checkpoints", allow_dir=True)
        log_artifact_if_exists(str(config_path), artifact_path="config")
    print("Adapter/best model in:", result["final_dir"])

In [ ]:
import json

if result is None:
    print("Nessun training eseguito (baseline zero-shot): nessun riepilogo.")
else:
    state = json.loads(Path(result["trainer_state"]).read_text())
    hist = state.get("log_history", [])
    _skip = {"eval_runtime", "eval_samples_per_second", "eval_steps_per_second"}
    eval_keys = sorted(
        {k for h in hist for k in h if k.startswith("eval_") and k not in _skip}
    )

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    train = [(h["step"], h["loss"]) for h in hist if "loss" in h]
    if train:
        xs, ys = zip(*train)
        axes[0].plot(xs, ys, color="#c1121f")
    axes[0].set(title="Train loss", xlabel="step", ylabel="loss")
    axes[0].grid(alpha=0.3)
    for key in eval_keys:
        pts = [(h["epoch"], h[key]) for h in hist if key in h and "epoch" in h]
        if pts:
            xs, ys = zip(*pts)
            axes[1].plot(xs, ys, marker="o", label=key.replace("eval_", ""))
    axes[1].set(title="Metriche eval per epoca", xlabel="epoch")
    axes[1].grid(alpha=0.3)
    if eval_keys:
        axes[1].legend(fontsize=8)
    plt.tight_layout()
    png = Path(result["output_dir"]) / "training_curves.png"
    plt.savefig(png, dpi=120, bbox_inches="tight")
    plt.show()

    max_steps = state.get("max_steps") or 0
    early_stopped = bool(max_steps) and state.get("global_step", 0) < max_steps
    print("Best checkpoint :", state.get("best_model_checkpoint"))
    print("Best metric     :", state.get("best_metric"), f"({config['finetuning'].get('metric_for_best_model', 'eval_loss')})")
    print("Epoche svolte   :", f"{state.get('epoch', 0):.2f}", "| step:", state.get("global_step"), "/", max_steps)
    print("Early stopping  :", "sì" if early_stopped else "no")
    if torch.cuda.is_available():
        print(f"Picco VRAM      : {torch.cuda.max_memory_allocated() / 1e9:.1f} GB")
    print("Curve salvate   :", png)